<a href="https://colab.research.google.com/github/Renata-Nascimento/Projeto-U02-CD/blob/main/notebooks/03_graficos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Renata-Nascimento/Projeto-U02-CD/blob/main/notebooks/03_graficos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Unidade 02 - Ciência de Dados (IMD/UFRN)
**Tema:** Para onde vai o dinheiro do Estado nos municípios do RN?

**Notebook 03:** Gráficos e Respostas às Perguntas Centrais
* **Responsável:** Brenda Gomes (Pessoa B)
* **Recorte:** 167 municípios do RN (período de 2020 a 2023)

---
### Objetivos deste Notebook
Responder às perguntas norteadoras do projeto por meio de visualizações analíticas estruturadas:
1. **P1 (Desigualdade):** Os repasses por habitante são equilibrados ou concentrados entre os municípios? Houve mudança no nível de desigualdade (Gini e Lorenz)?
2. **P2 (Ricos x Pobres):** O dinheiro do Estado chega mais aos municípios com maior PIB ou atua como equalizador social para municípios com maior pobreza?
3. **P3 (Região e Porte):** Como os repasses se distribuem entre as mesorregiões do RN e entre as diferentes faixas de população?
4. **Evolução Temporal (2020-2023):** Houve ganho real nas transferências estaduais ou apenas reposição inflacionária? Qual a composição por tipo de imposto?


## 1. Configuração do Ambiente e Carregamento dos Dados

Importamos as bibliotecas necessárias e configuramos o estilo visual padrão para os gráficos (Seaborn/Matplotlib).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuração estética dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

# Caminho dos dados: lê direto do GitHub ou do diretório local
URL_DADOS = "https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/base_municipios_2020_2023.csv"
PATH_LOCAL = "../saida/base_municipios_2020_2023.csv"

if os.path.exists(PATH_LOCAL):
    df = pd.read_csv(PATH_LOCAL)
    print("Base carregada do arquivo local:", PATH_LOCAL)
else:
    df = pd.read_csv(URL_DADOS)
    print("Base carregada diretamente do GitHub.")

# Padronização das categorias de porte demográfico baseada no Censo 2022
bins_porte = [0, 5000, 10000, 20000, 50000, float("inf")]
labels_porte = ["até 5 mil", "5 a 10 mil", "10 a 20 mil", "20 a 50 mil", "mais de 50 mil"]
df["porte_cat"] = pd.cut(df["populacao"], bins=bins_porte, labels=labels_porte)

print(f"Total de registros: {len(df)} linhas ({df['municipio'].nunique()} municípios ao longo de {df['ano'].nunique()} anos).")
df[["municipio", "ano", "rep_per_capita", "pib_per_capita", "pct_baixa_renda"]].head()


## 2. Pergunta 1: Desigualdade na Distribuição dos Recursos

Analisamos a distribuição do repasse real por habitante (`rep_per_capita`) e calculamos a **Curva de Lorenz** e o **Coeficiente de Gini** ano a ano para verificar a concentração dos recursos estaduais.


In [ ]:
# 2.1 Histograma e Boxplot do Repasse por Habitante (2023)
df_2023 = df[df["ano"] == 2023].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma com KDE
sns.histplot(df_2023["rep_per_capita"], bins=30, kde=True, color="#2b5c8f", ax=axes[0])
axes[0].axvline(df_2023["rep_per_capita"].median(), color="#d95f02", linestyle="--", linewidth=1.5,
                label=f"Mediana: R$ {df_2023['rep_per_capita'].median():.2f}")
axes[0].axvline(df_2023["rep_per_capita"].mean(), color="#7570b3", linestyle=":", linewidth=1.5,
                label=f"Média: R$ {df_2023['rep_per_capita'].mean():.2f}")
axes[0].set_title("Histograma: Repasse por Habitante (2023)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Repasse Real por Habitante (R$ de 2023)", fontsize=11)
axes[0].set_ylabel("Quantidade de Municípios", fontsize=11)
axes[0].legend(frameon=True)

# Boxplot
sns.boxplot(x=df_2023["rep_per_capita"], color="#80b1d3", ax=axes[1], flierprops={"markerfacecolor":"#e41a1c", "markersize":6})
axes[1].set_title("Boxplot: Repasse por Habitante e Outliers (2023)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Repasse Real por Habitante (R$ de 2023)", fontsize=11)

# Destaque para os maiores outliers
top_outliers = df_2023.nlargest(3, "rep_per_capita")
for _, row in top_outliers.iterrows():
    texto_rotulo = row["municipio"] + "\n(R$ " + f"{row['rep_per_capita']:.0f}" + ")"
    axes[1].annotate(texto_rotulo, xy=(row["rep_per_capita"], 0), xytext=(row["rep_per_capita"], -0.2),
                     arrowprops=dict(arrowstyle="->", color="#e41a1c"), fontsize=9, ha="center")

plt.tight_layout()
plt.show()


> **Interpretação:**
> A distribuição do repasse per capita apresenta acentuada assimetria à direita: a grande maioria dos municípios potiguares situa-se entre R$ 400 e R$ 900 anuais por habitante (mediana de R$ 610,67). A média estadual (R$ 917,74) é puxada para cima por municípios atípicos com altíssima arrecadação pontual de royalties e cotas de energia/petróleo, com destaque para Guamaré (R$ 10.780/hab) e São Bento do Norte (R$ 7.798/hab).


In [ ]:
# 2.2 Curva de Lorenz e Coeficiente de Gini por Ano (2020 a 2023)
def calcular_gini(valores):
    """Calcula o Coeficiente de Gini de um array unidimensional."""
    arr = np.asarray(valores, dtype=float).flatten()
    if np.amin(arr) < 0:
        arr -= np.amin(arr)
    arr += 1e-9
    arr = np.sort(arr)
    n = arr.shape[0]
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * arr)) / (n * np.sum(arr))

def curva_lorenz(valores):
    """Retorna os pontos da Curva de Lorenz."""
    arr = np.sort(np.asarray(valores, dtype=float))
    cum_vals = np.cumsum(arr)
    cum_vals = np.insert(cum_vals, 0, 0)
    cum_pct = cum_vals / cum_vals[-1]
    pop_pct = np.linspace(0, 1, len(cum_pct))
    return pop_pct, cum_pct

plt.figure(figsize=(8, 7))
cores_anos = {2020: "#1b9e77", 2021: "#d95f02", 2022: "#7570b3", 2023: "#e7298a"}

# Linha de perfeita igualdade (45 graus)
plt.plot([0, 1], [0, 1], color="#333333", linestyle="--", linewidth=1.5, label="Perfeita Igualdade (Gini = 0)")

for ano in sorted(df["ano"].unique()):
    sub = df[df["ano"] == ano]["rep_per_capita"].values
    gini_val = calcular_gini(sub)
    x_lor, y_lor = curva_lorenz(sub)
    plt.plot(x_lor, y_lor, label=f"Ano {ano} (Gini = {gini_val:.3f})", color=cores_anos[ano], linewidth=2)

plt.title("Curva de Lorenz: Desigualdade do Repasse por Habitante (2020-2023)", fontsize=13, fontweight="bold")
plt.xlabel("Proporção Acumulada de Municípios", fontsize=11)
plt.ylabel("Proporção Acumulada de Repasses per Capita", fontsize=11)
plt.legend(frameon=True, loc="upper left")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.show()


> **Interpretação:**
> O Coeficiente de Gini do repasse per capita manteve-se praticamente inalterado ao longo do quadriênio, oscilando entre 0,423 e 0,432 (0,425 em 2023). A Curva de Lorenz evidencia uma concentração estrutural persistente: os 50% dos municípios potiguares que menos recebem respondem por apenas cerca de 23% do somatório dos repasses per capita, revelando rigidez na partilha dos tributos estaduais.


## 3. Pergunta 2: Repasses em Municípios Ricos x Pobres

Investigamos se os repasses estaduais premiam economias maiores ou compensam municípios com maior contingente de pobreza cadastrada no CadÚnico.


In [ ]:
# 3.1 Gráficos de Dispersão: Repasse per Capita x PIB per Capita e x % Baixa Renda
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dispersão 1: Repasse per capita x PIB per capita (2023)
sns.regplot(data=df_2023, x="pib_per_capita", y="rep_per_capita",
            scatter_kws={"alpha": 0.7, "color": "#2b5c8f"}, line_kws={"color": "#e41a1c", "linewidth": 1.5}, ax=axes[0])
corr_pib = df_2023["rep_per_capita"].corr(df_2023["pib_per_capita"])
axes[0].set_title(f"Repasse per Capita x PIB per Capita (2023)\nCorrelação linear: r = {corr_pib:.2f}", fontsize=13, fontweight="bold")
axes[0].set_xlabel("PIB per Capita (R$ de 2023)", fontsize=11)
axes[0].set_ylabel("Repasse per Capita (R$ de 2023)", fontsize=11)

# Destaque para municípios com maior PIB per capita
for _, row in df_2023.nlargest(4, "pib_per_capita").iterrows():
    axes[0].annotate(row["municipio"], xy=(row["pib_per_capita"], row["rep_per_capita"]),
                     xytext=(10, 0), textcoords="offset points", fontsize=8, fontweight="bold")

# Dispersão 2: Repasse per capita x % Baixa Renda no CadÚnico (2023)
sns.regplot(data=df_2023, x="pct_baixa_renda", y="rep_per_capita",
            scatter_kws={"alpha": 0.7, "color": "#1b9e77"}, line_kws={"color": "#d95f02", "linewidth": 1.5}, ax=axes[1])
corr_baixa = df_2023["rep_per_capita"].corr(df_2023["pct_baixa_renda"])
axes[1].set_title(f"Repasse per Capita x % Baixa Renda no CadÚnico (2023)\nCorrelação linear: r = {corr_baixa:.2f}", fontsize=13, fontweight="bold")
axes[1].set_xlabel("População de Baixa Renda no CadÚnico (%)", fontsize=11)
axes[1].set_ylabel("Repasse per Capita (R$ de 2023)", fontsize=11)

plt.tight_layout()
plt.show()


> **Interpretação:**
> Há correlação linear positiva muito forte (r = 0,81) entre o PIB per capita e o repasse per capita, evidenciando que municípios de economias mais dinâmicas retêm as maiores parcelas de recursos por habitante. Por outro lado, a correlação com a proporção de famílias de baixa renda é quase nula (r = 0,13), mostrando que as transferências estaduais não possuem desenho compensatório ou focalizado em vulnerabilidade social.


In [ ]:
# 3.2 Mapa de Calor de Correlação (Matriz Socioeconômica e Fiscal)
cols_corr = ["rep_per_capita", "pib_per_capita", "pct_baixa_renda", "populacao", "rep_total_real", "pib_real_mil"]
nomes_corr = ["Repasse per Capita", "PIB per Capita", "% Baixa Renda", "População", "Repasse Total Real", "PIB Total Real"]

matriz_corr = df_2023[cols_corr].corr()
matriz_corr.columns = nomes_corr
matriz_corr.index = nomes_corr

plt.figure(figsize=(8, 6))
sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={"label": "Coeficiente de Pearson (r)"})
plt.title("Mapa de Calor: Correlações Fiscais e Socioeconômicas (2023)", fontsize=13, fontweight="bold", pad=12)
plt.show()


> **Interpretação:**
> O mapa de calor confirma que os volumes agregados de repasse e PIB acompanham diretamente a escala populacional (r > 0,94). No entanto, sob a ótica per capita, o repasse se desvincula da população (r = -0,05) e se atrela fortemente ao PIB per capita (r = 0,81), sem apresentar correlação protetiva com o índice de famílias de baixa renda (r = 0,13).


## 4. Pergunta 3: Disparidades por Mesorregião e Porte Populacional

Avaliamos como os repasses per capita variam entre as quatro mesorregiões geográficas do Rio Grande do Norte e entre diferentes classes de tamanho populacional.


In [ ]:
# 4.1 Boxplot por Mesorregião (2023)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Visão completa
sns.boxplot(data=df_2023, x="mesorregiao", y="rep_per_capita", hue="mesorregiao", palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Repasse per Capita por Mesorregião (Escala Total)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Mesorregião", fontsize=11)
axes[0].set_ylabel("Repasse per Capita (R$ de 2023)", fontsize=11)

# Visão sem outliers para evidenciar as medianas regionais
sns.boxplot(data=df_2023, x="mesorregiao", y="rep_per_capita", hue="mesorregiao", palette="Set2", legend=False, showfliers=False, ax=axes[1])
axes[1].set_title("Repasse per Capita por Mesorregião (Sem Outliers Extremos)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Mesorregião", fontsize=11)
axes[1].set_ylabel("Repasse per Capita (R$ de 2023)", fontsize=11)

plt.tight_layout()
plt.show()


> **Interpretação:**
> A mesorregião Central Potiguar apresenta a maior média (R$ 1.433/hab) e extrema dispersão, impulsionada por municípios salineiros, petrolíferos e eólicos da Costa Branca. O Oeste Potiguar detém a maior mediana regional (R$ 683/hab), enquanto o Agreste Potiguar (R$ 477/hab) e o Leste Potiguar (R$ 472/hab, desconsiderando polos urbanos) apresentam os menores repasses medianos por habitante.


In [ ]:
# 4.2 Boxplot por Faixa de População (Porte Demográfico em 2023)
plt.figure(figsize=(11, 5))
sns.boxplot(data=df_2023, x="porte_cat", y="rep_per_capita",
            order=labels_porte, hue="porte_cat", palette="Blues_d", legend=False, showfliers=False)
plt.title("Repasse por Habitante por Faixa Populacional (2023 - Sem Outliers)", fontsize=13, fontweight="bold")
plt.xlabel("Faixa de População (Censo 2022)", fontsize=11)
plt.ylabel("Repasse Real por Habitante (R$ de 2023)", fontsize=11)

# Adicionar número de municípios por porte
contagem_porte = df_2023["porte_cat"].value_counts()
for i, p in enumerate(labels_porte):
    qtd = contagem_porte.get(p, 0)
    plt.text(i, plt.ylim()[0] + 20, f"n = {qtd}", ha="center", fontsize=10, color="#555555")

plt.tight_layout()
plt.show()


> **Interpretação:**
> Municípios de menor contingente demográfico ("até 5 mil habitantes") alcançam a maior mediana per capita (R$ 824/hab), decorrente do efeito de divisão de cotas mínimas por uma população bastante reduzida. Cidades de porte intermediário (10 a 20 mil e 20 a 50 mil habitantes) apresentam as menores medianas (cerca de R$ 460 a R$ 480/hab), configurando uma curva em formato de "U" invertido no ganho por habitante.


## 5. Evolução Temporal dos Repasses (2020 a 2023)

Comparamos a trajetória do total repassado sob a ótica nominal versus real (ajustado pelo IPCA) e inspecionamos o peso de cada imposto estadual.


In [ ]:
# 5.1 Repasses Totais: Comparativo com e sem Inflação (Nominal vs Real)
totais_anuais = df.groupby("ano")[["rep_total", "rep_total_real"]].sum().reset_index()
totais_anuais["rep_total_bi"] = totais_anuais["rep_total"] / 1e9
totais_anuais["rep_total_real_bi"] = totais_anuais["rep_total_real"] / 1e9

fig, ax = plt.subplots(figsize=(10, 5))
largura = 0.35
x = np.arange(len(totais_anuais["ano"]))

barras_nominais = ax.bar(x - largura/2, totais_anuais["rep_total_bi"], largura,
                         label="Repasse Nominal (R$ correntes)", color="#9ecae1")
barras_reais = ax.bar(x + largura/2, totais_anuais["rep_total_real_bi"], largura,
                      label="Repasse Real (R$ constantes de 2023)", color="#2171b5")

ax.set_title("Evolução dos Repasses Totais: Nominal vs Real pelo IPCA (2020-2023)", fontsize=13, fontweight="bold")
ax.set_xlabel("Ano", fontsize=11)
ax.set_ylabel("Repasses Totais (em Bilhões de R$)", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(totais_anuais["ano"])
ax.legend(frameon=True)

# Inserir valores no topo das barras
for bar in barras_nominais:
    h = bar.get_height()
    ax.annotate(f"R$ {h:.2f} bi", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

for bar in barras_reais:
    h = bar.get_height()
    ax.annotate(f"R$ {h:.2f} bi", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9, fontweight="bold")

ax.set_ylim(0, max(totais_anuais["rep_total_real_bi"]) * 1.15)
plt.tight_layout()
plt.show()


> **Interpretação:**
> Em termos nominais, as transferências estaduais aos municípios saltaram de R$ 1,63 bilhão em 2020 para R$ 2,43 bilhões em 2023 (+48,6%). No entanto, após o desconto da inflação medida pelo IPCA, o ganho de poder de compra real foi de 20,1% (de R$ 2,02 bi para R$ 2,43 bi), registrando estagnação real entre 2021 e 2022 devido ao pico de inflação no período.


In [ ]:
# 5.2 Evolução por Tipo de Imposto Repassado (2020 a 2023)
tributos = ["rep_icms", "rep_ipva", "rep_royalties", "rep_ipi"]
nomes_tributos = ["ICMS", "IPVA", "Royalties", "IPI-Exportação"]
cores_trib = ["#08519c", "#3182bd", "#6baed6", "#bdd7e7"]

totais_trib = df.groupby("ano")[tributos].sum() / 1e9
totais_trib.columns = nomes_tributos

ax = totais_trib.plot(kind="bar", stacked=True, figsize=(10, 5), color=cores_trib, rot=0)
plt.title("Composição e Evolução dos Repasses por Tributo (2020-2023)", fontsize=13, fontweight="bold")
plt.xlabel("Ano", fontsize=11)
plt.ylabel("Valor Arrecadado e Repassado (Bilhões de R$)", fontsize=11)
plt.legend(title="Tributo", frameon=True)

# Calcular % do ICMS no último ano
icms_pct_2023 = (totais_trib.loc[2023, "ICMS"] / totais_trib.loc[2023].sum()) * 100
plt.annotate(f"ICMS representa {icms_pct_2023:.1f}% do total em 2023",
             xy=(3, totais_trib.loc[2023].sum()), xytext=(2.2, totais_trib.loc[2023].sum() + 0.15),
             arrowprops=dict(arrowstyle="->", color="#08519c"), fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()


> **Interpretação:**
> O ICMS é o tributo hegemônico nas transferências do Governo do RN, concentrando cerca de 85,5% do total repassado em 2023 (R$ 2,08 bi de R$ 2,43 bi). O IPVA consolida-se como a segunda fonte mais representativa (12,6%), enquanto Royalties e IPI-Exportação desempenham papel marginal no montante global estadual, embora os royalties sejam o principal motor fiscal para os municípios produtores.


## 6. Síntese Executiva dos Resultados (Roteiro para a Reunião)

Tabela consolidada com os principais indicadores derivados para alimentar o **Resumo de 1 Página** da equipe:

| Pergunta / Dimensão | Indicador Chave | Conclusão Principal |
|---|---|---|
| **P1. Desigualdade** | Gini = 0,425 (estável 2020-2023) | Forte assimetria com outliers extremos (Guamaré R$ 10,7 mil/hab vs Extremoz R$ 171/hab). |
| **P2. Ricos x Pobres** | Corr(PIB, Repasse) = +0,81 | Repasses são pró-cíclicos com o PIB e neutros em relação à vulnerabilidade social (CadÚnico). |
| **P3. Regiões** | Central lidera média; Oeste lidera mediana | Central tem forte dispersão (petróleo/energia); Leste e Agreste têm menores medianas. |
| **P3. Portes** | Até 5 mil hab: R$ 824/hab (mediana) | Municípios com populações mínimas se beneficiam de cotas fixas; cidades médias têm menor valor. |
| **Evolução Temporal** | Ganho real de 20,1% no quadriênio | Ilusão nominal de +48,6% atenuada pela inflação do IPCA, com hegemonia de 85,5% do ICMS. |
